In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import tensorflow.keras as keras
from sklearn.datasets import fetch_california_housing

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

2026-03-18 14:53:26.461073: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-18 14:53:26.461354: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-18 14:53:26.493863: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-18 14:53:27.219770: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation or

## L2-Regularisierung

L2-Regularisierung bestraft große Gewichte im Netzwerk, wodurch ein einzelnes Feature nicht übermäßig an Bedeutung gewinnt und Overfitting reduziert wird.

In [ ]:
SEED = 42
SF_COORDS = (37.7749, -122.4194)
LA_COORDS = (34.0522, -118.2437)
SJ_COORDS = (37.3362, -121.8833)


def prepare_dataset():
    data = fetch_california_housing()
    df = pd.DataFrame(data=data.data, columns=data.feature_names)
    df[data.target_names[0]] = data.target

    mask_cutoff = (
        (df["MedHouseVal"] < df["MedHouseVal"].max())
        & (df["HouseAge"] < df["HouseAge"].max())
        & (df["MedInc"] < df["MedInc"].max())
    )
    df_clean = df[mask_cutoff].copy()

    clip_cols = ["AveRooms", "AveBedrms", "AveOccup", "Population"]
    for col in clip_cols:
        lower_bound = df_clean[col].quantile(0.01)
        upper_bound = df_clean[col].quantile(0.99)
        df_clean[col] = df_clean[col].clip(lower=lower_bound, upper=upper_bound)

    df_clean["BedrmsPerRoom"] = df_clean["AveBedrms"] / df_clean["AveRooms"]

    df_clean["MedInc"] = np.log1p(df_clean["MedInc"])
    df_clean["Population"] = np.log1p(df_clean["Population"])
    df_clean["AveBedrms"] = np.log1p(df_clean["AveBedrms"])

    df_clean["Dist_to_SF"] = np.sqrt(
        (df_clean["Latitude"] - SF_COORDS[0]) ** 2 + (df_clean["Longitude"] - SF_COORDS[1]) ** 2
    )
    df_clean["Dist_to_LA"] = np.sqrt(
        (df_clean["Latitude"] - LA_COORDS[0]) ** 2 + (df_clean["Longitude"] - LA_COORDS[1]) ** 2
    )
    df_clean["Dist_to_SJ"] = np.sqrt(
        (df_clean["Latitude"] - SJ_COORDS[0]) ** 2 + (df_clean["Longitude"] - SJ_COORDS[1]) ** 2
    )

    X = df_clean.drop("MedHouseVal", axis=1)
    y = df_clean["MedHouseVal"].copy()

    X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
        X, y, test_size=0.2, random_state=SEED
    )

    scaler = sklearn.preprocessing.StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return df_clean, X_train, X_test, y_train, y_test, X_train_scaled, X_test_scaled


df_clean, X_train, X_test, y_train, y_test, X_train_scaled, X_test_scaled = prepare_dataset()
print(f"Neue Datensatzgröße: {df_clean.shape}")

Neue Datensatzgröße: (18570, 13)


In [ ]:

def build_model(regularization):
    model = keras.Sequential([
        keras.layers.Input(shape=(X_train_scaled.shape[1],)),
        keras.layers.Dense(128, activation="relu", kernel_regularizer=keras.regularizers.l2(regularization)),
        keras.layers.BatchNormalization(),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(64, activation="relu", kernel_regularizer=keras.regularizers.l2(regularization)),
        keras.layers.BatchNormalization(),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.BatchNormalization(),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(1, activation="linear"),
    ])
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model


model = build_model(0.001)
model.fit(
    X_train_scaled,
    y_train,
    epochs=150,
    batch_size=64,
    validation_split=0.2,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5),
    ],
    verbose=0,
)

test_scores = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"MSE: {test_scores[0]:.2f}")
print(f"MAE: {test_scores[1]:.2f}")

Epoch 1/150


2026-03-18 14:53:28.089226: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


186/186 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 3.0119 - mae: 1.3923 - val_loss: 1.2283 - val_mae: 0.8231 - learning_rate: 0.0010
Epoch 2/150
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1.0848 - mae: 0.7581 - val_loss: 0.5238 - val_mae: 0.4537 - learning_rate: 0.0010
Epoch 3/150
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.7973 - mae: 0.6310 - val_loss: 0.3953 - val_mae: 0.3807 - learning_rate: 0.0010
Epoch 4/150
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.6611 - mae: 0.5641 - val_loss: 0.3771 - val_mae: 0.3665 - learning_rate: 0.0010
Epoch 5/150
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.5778 - mae: 0.5190 - val_loss: 0.3558 - val_mae: 0.3586 - learning_rate: 0.0010
Epoch 6/150
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.5357 - mae: 0.4932 - val_loss: 0.3472 - val_mae: 0.3497 - learning_rate: 0.0010
Epoch 7/150
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.5006 - mae: 0.4708 - val_loss: 0.3463 - val_mae: 0.3532 - learning_rate: 0.0010
Epoch 8/150

## Feinabstimmung der L2-Regularisierung

Im nächsten Schritt werden verschiedene Regularisierungsstärken geprüft: 0.1, 0.01, 0.001 und 0.0001.

In [ ]:
results = {}

for regularization in [0.1, 0.01, 0.001, 0.0001]:
    print(f"\n--- Starte Training mit L2 = {regularization} ---")
    model = build_model(regularization)

    model.fit(
        X_train_scaled,
        y_train,
        epochs=150,
        batch_size=64,
        validation_split=0.2,
        callbacks=[
            keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5),
        ],
        verbose=0,
    )

    test_scores = model.evaluate(X_test_scaled, y_test, verbose=0)
    print(f"Regularization: {regularization}")
    print(f"MSE: {test_scores[0]:.2f}")
    print(f"MAE: {test_scores[1]:.2f}")
    results[regularization] = (np.round(test_scores[0], decimals=2), np.round(test_scores[1], decimals=2))

for regularization, values in results.items():
    print(f"L2 Reg: {regularization} - {values}")


--- Starte Training mit L2 = 0.1---
117/117 ━━━━━━━━━━━━━━━━━━━━ 0s 526us/step - loss: 0.2312 - mae: 0.3173
Regularization: 0.1
MSE: 0.23
MAE: 0.32

--- Starte Training mit L2 = 0.01---
117/117 ━━━━━━━━━━━━━━━━━━━━ 0s 590us/step - loss: 0.2207 - mae: 0.3059
Regularization: 0.01
MSE: 0.22
MAE: 0.31

--- Starte Training mit L2 = 0.001---
117/117 ━━━━━━━━━━━━━━━━━━━━ 0s 525us/step - loss: 0.2147 - mae: 0.3011
Regularization: 0.001
MSE: 0.21
MAE: 0.30

--- Starte Training mit L2 = 0.0001---
117/117 ━━━━━━━━━━━━━━━━━━━━ 0s 568us/step - loss: 0.2106 - mae: 0.3042
Regularization: 0.0001
MSE: 0.21
MAE: 0.30
L2 Reg: 0.1 - (np.float64(0.23), np.float64(0.32))
L2 Reg: 0.01 - (np.float64(0.22), np.float64(0.31))
L2 Reg: 0.001 - (np.float64(0.21), np.float64(0.3))
L2 Reg: 0.0001 - (np.float64(0.21), np.float64(0.3))
